# Test: GPT-4o — TAMAS Benchmark Baseline (API)

OpenAI API-based. Used for reproducing TAMAS baselines on flat AutoGen.

**Prerequisites:** `OPENAI_API_KEY` set in environment or `.env`

In [ ]:
import sys
sys.path.insert(0, '/storage/data/AgenticCyOps_Private')

from dotenv import load_dotenv
load_dotenv('/storage/data/AgenticCyOps_Private/.env')

from models.utils import GPT4o

model = GPT4o()
print('Model config:')
model.get_config()

## 1. Health Check

In [ ]:
assert model.health_check(), 'API key invalid or API unreachable!'
print('Health check passed — API key valid')

## 2. Chat Completions

In [ ]:
messages = [
    {'role': 'system', 'content': 'You are a SOC analyst. Be concise.'},
    {'role': 'user', 'content': 'What is a lateral movement attack? One sentence.'}
]

# 2a. Basic chat
resp = model.chat(messages, max_tokens=100)
print('Basic chat:', resp.choices[0].message.content)

In [ ]:
# 2b. Deterministic
resp1 = model.chat_deterministic(messages, max_tokens=100)
resp2 = model.chat_deterministic(messages, max_tokens=100)
print('Deterministic r1:', resp1.choices[0].message.content[:80])
print('Deterministic r2:', resp2.choices[0].message.content[:80])
print('Match:', resp1.choices[0].message.content == resp2.choices[0].message.content)

In [ ]:
# 2c. Creative
resp = model.chat_creative(messages, max_tokens=100)
print('Creative:', resp.choices[0].message.content)

In [ ]:
# 2d. Streaming
stream = model.chat(messages, max_tokens=100, stream=True)
print('Streaming: ', end='')
for chunk in stream:
    delta = chunk.choices[0].delta.content
    if delta:
        print(delta, end='', flush=True)
print()

## 3. Tool / Function Calling

In [ ]:
tools = [
    {
        'type': 'function',
        'function': {
            'name': 'query_siem',
            'description': 'Search SIEM logs for security events',
            'parameters': {
                'type': 'object',
                'properties': {
                    'query': {'type': 'string', 'description': 'Search query'},
                    'time_range': {'type': 'string', 'description': 'Time range'},
                },
                'required': ['query']
            }
        }
    },
    {
        'type': 'function',
        'function': {
            'name': 'isolate_host',
            'description': 'Isolate a host from the network',
            'parameters': {
                'type': 'object',
                'properties': {
                    'hostname': {'type': 'string'},
                    'reason': {'type': 'string'},
                },
                'required': ['hostname', 'reason']
            }
        }
    }
]

In [ ]:
# 3a. Auto tool choice
tc_messages = [
    {'role': 'system', 'content': 'You are a SOC analyst.'},
    {'role': 'user', 'content': 'Search SIEM for failed logins from 10.0.5.12 in the last hour.'}
]
resp = model.tool_call(tc_messages, tools)
tc = resp.choices[0].message.tool_calls
print(f'Auto: {len(tc)} call(s)')
for c in tc:
    print(f'  {c.function.name}({c.function.arguments})')

In [ ]:
# 3b. Required tool choice
resp = model.tool_call_required(tc_messages, tools)
tc = resp.choices[0].message.tool_calls
print(f'Required: {len(tc)} call(s)')
for c in tc:
    print(f'  {c.function.name}({c.function.arguments})')

In [ ]:
# 3c. Specific tool
resp = model.tool_call_specific(tc_messages, tools, 'isolate_host')
tc = resp.choices[0].message.tool_calls
print(f'Specific: {len(tc)} call(s)')
for c in tc:
    print(f'  {c.function.name}({c.function.arguments})')

## 4. Structured Output

In [ ]:
# 4a. JSON mode
json_messages = [
    {'role': 'system', 'content': 'Respond with JSON only.'},
    {'role': 'user', 'content': 'Classify: "Multiple failed SSH logins from 10.0.5.12". Return {"severity": str, "category": str, "confidence": float}'}
]
resp = model.chat_json(json_messages, temperature=0.0, max_tokens=200)
print('JSON mode:', resp.choices[0].message.content)

In [ ]:
# 4b. JSON schema
schema = {
    'type': 'object',
    'properties': {
        'severity': {'type': 'string', 'enum': ['low', 'medium', 'high', 'critical']},
        'category': {'type': 'string'},
        'confidence': {'type': 'number', 'minimum': 0, 'maximum': 1}
    },
    'required': ['severity', 'category', 'confidence'],
    'additionalProperties': False
}
resp = model.chat_json_schema(json_messages, schema, temperature=0.0, max_tokens=200)
print('JSON schema:', resp.choices[0].message.content)

## 5. Batch Chat

In [ ]:
batches = [
    [{'role': 'user', 'content': 'What is phishing? One sentence.'}],
    [{'role': 'user', 'content': 'What is ransomware? One sentence.'}],
    [{'role': 'user', 'content': 'What is a zero-day? One sentence.'}],
]
results = model.batch_chat(batches, max_tokens=80)
for i, r in enumerate(results):
    print(f'Batch {i}: {r.choices[0].message.content}')

## 6. Token Usage

In [ ]:
resp = model.chat(messages, max_tokens=100)
usage = resp.usage
print(f'Prompt tokens:     {usage.prompt_tokens}')
print(f'Completion tokens: {usage.completion_tokens}')
print(f'Total tokens:      {usage.total_tokens}')

## 7. Get Config

In [ ]:
import json
print(json.dumps(model.get_config(), indent=2))

## Summary

All tests passed if no cells raised exceptions above.